# Experiment: Inspect Disk Backed ChangBai CC Cube

Objective:
- Open an existing disk-backed HDF5 cube with shape `(ndate, nshot, nt, noffset)`.
- Inspect metadata, slice one fixed shot across all dates, and make quick plots.
- Keep notebook responsibility limited to HDF5 reading and visualization.

Success criteria:
- The notebook opens an existing `cc_cube_by_date.h5` without touching the original `.mat` files.
- Example cells can read `[:, ishot, :, :]`, inspect metadata, and generate summary plots.


In [ ]:
# Setup: imports, plotting defaults, and notebook paths
from __future__ import annotations

from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.sans-serif"] = ["Times New Roman"]
plt.rcParams["figure.dpi"] = 120

ROOT_DIR = Path("../outputs/cc_tsi_plot_outputs_ChangBai").resolve()
OUTPUT_H5 = ROOT_DIR / "cc_cube_by_date.h5"
META_CSV = ROOT_DIR / "cc_cube_index.csv"
SHOT_INDEX = 10

ROOT_DIR, OUTPUT_H5


## Plan

- Open the existing HDF5 cube and print global shape plus chunk layout.
- Read the CSV index if it exists, so the date-to-file mapping remains visible.
- Slice one fixed shot across all dates with `h5f["cc"][:, SHOT_INDEX, :, :]`.
- Plot one date / one shot image and simple cross-date statistics.


In [ ]:
# Helpers: HDF5 summary and optional CSV index loading
def summarize_hdf5(output_h5: Path) -> dict[str, object]:
    with h5py.File(output_h5, "r") as h5f:
        return {
            "cc_shape": tuple(int(x) for x in h5f["cc"].shape),
            "cc_dtype": str(h5f["cc"].dtype),
            "date_count": int(h5f["date_labels"].shape[0]),
            "root_dir": h5f.attrs["root_dir"],
            "created_at": h5f.attrs.get("created_at", ""),
            "chunk_mode": h5f.attrs.get("chunk_mode", ""),
            "chunk_shape": tuple(int(x) for x in h5f["cc"].chunks),
        }


def load_index_csv(meta_csv: Path) -> pd.DataFrame | None:
    if not meta_csv.exists():
        print(f"Index CSV not found: {meta_csv}")
        return None
    return pd.read_csv(meta_csv)


if not OUTPUT_H5.exists():
    raise FileNotFoundError(
        f"HDF5 file not found: {OUTPUT_H5}. Run `uv run python tools/build_cc_h5_cube.py ...` first."
    )


## Inspect Global Metadata

This stage only opens the existing HDF5 cube and its optional CSV index.


In [ ]:
summary = summarize_hdf5(OUTPUT_H5)
summary


In [ ]:
index_df = load_index_csv(META_CSV)
if index_df is not None:
    display(index_df.head(20))


## Re-Open The Cube And Inspect Slices

These examples operate directly on the HDF5-backed dataset.


In [ ]:
with h5py.File(OUTPUT_H5, "r") as h5f:
    cc_shape = tuple(int(x) for x in h5f["cc"].shape)
    date_labels = [x.decode("utf-8") if isinstance(x, bytes) else str(x) for x in h5f["date_labels"][:]]
    print("cc shape:", cc_shape)
    print("chunk shape:", tuple(int(x) for x in h5f["cc"].chunks))
    print("first dates:", date_labels[: min(5, len(date_labels))])
    first_cube = h5f["cc"][0]
    print("first date cube shape:", first_cube.shape)


In [ ]:
# Example 1: show one date / one shot image
DATE_INDEX = 0

with h5py.File(OUTPUT_H5, "r") as h5f:
    if SHOT_INDEX >= h5f["cc"].shape[1]:
        raise IndexError(f"SHOT_INDEX={SHOT_INDEX} out of range for nshot={h5f['cc'].shape[1]}")
    cc_slice = h5f["cc"][DATE_INDEX, SHOT_INDEX]
    date_label = h5f["date_labels"][DATE_INDEX]
    date_label = date_label.decode("utf-8") if isinstance(date_label, bytes) else str(date_label)

vmax = float(np.percentile(np.abs(cc_slice), 99))
vmax = 1.0 if vmax == 0 else vmax
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(cc_slice, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax, origin="lower")
ax.set_title(f"Date={date_label}, shot={SHOT_INDEX}")
ax.set_xlabel("Offset index")
ax.set_ylabel("Time index")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


In [ ]:
# Example 2: load one shot across all dates
with h5py.File(OUTPUT_H5, "r") as h5f:
    if SHOT_INDEX >= h5f["cc"].shape[1]:
        raise IndexError(f"SHOT_INDEX={SHOT_INDEX} out of range for nshot={h5f['cc'].shape[1]}")
    shot_cube = h5f["cc"][:, SHOT_INDEX, :, :]
    date_labels = [x.decode("utf-8") if isinstance(x, bytes) else str(x) for x in h5f["date_labels"][:]]

print("shot cube shape:", shot_cube.shape)
shot_cube.shape


In [ ]:
# Example 3: summary statistics for one shot across all dates
mean_trace_by_date = shot_cube.mean(axis=(1, 2))
shot_energy_by_date = np.sqrt((shot_cube ** 2).mean(axis=(1, 2)))
summary_df = pd.DataFrame(
    {
        "date_label": date_labels,
        "mean_amplitude": mean_trace_by_date,
        "rms_amplitude": shot_energy_by_date,
    }
)
display(summary_df.head(20))


In [ ]:
# Example 4: quick image browser for one fixed shot across dates
DATE_INDEX = 0
date_label = date_labels[DATE_INDEX]
cc_slice = shot_cube[DATE_INDEX]

vmax = float(np.percentile(np.abs(cc_slice), 99))
vmax = 1.0 if vmax == 0 else vmax
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(cc_slice, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax, origin="lower")
ax.set_title(f"Shot={SHOT_INDEX}, date={date_label}")
ax.set_xlabel("Offset index")
ax.set_ylabel("Time index")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


In [ ]:
# Example 5: date-wise mean / rms plot for the selected shot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(date_labels, mean_trace_by_date, marker="o", label="mean")
ax.plot(date_labels, shot_energy_by_date, marker="s", label="rms")
ax.set_title(f"Shot {SHOT_INDEX} summary across dates")
ax.set_xlabel("Date folder")
ax.set_ylabel("Amplitude")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# Example 6: global date-wise mean / std and a quick sanity plot
with h5py.File(OUTPUT_H5, "r") as h5f:
    cc = h5f["cc"]
    date_labels = [x.decode("utf-8") if isinstance(x, bytes) else str(x) for x in h5f["date_labels"][:]]
    date_mean = np.asarray([cc[i].mean() for i in range(cc.shape[0])], dtype=float)
    date_std = np.asarray([cc[i].std() for i in range(cc.shape[0])], dtype=float)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(date_labels, date_mean, marker="o", label="mean")
ax.plot(date_labels, date_std, marker="s", label="std")
ax.set_title("Date-wise global summary statistics")
ax.set_xlabel("Date folder")
ax.set_ylabel("Amplitude")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
plt.show()


## Results

- The notebook depends only on `cc_cube_by_date.h5` and optional `cc_cube_index.csv`.
- Fixed-shot cross-date access should be efficient because the CLI writes a shot-oriented chunk layout by default.
- The original `.mat` to HDF5 conversion is intentionally removed from the notebook.


## Next Steps

- If you need a fresh cube, rerun `uv run python tools/build_cc_h5_cube.py ...` outside the notebook.
- Add downstream analysis cells that compute cross-date changes for fixed shots or offsets.
- If you later want interactivity, add widgets around `SHOT_INDEX` and `DATE_INDEX`.
